# Chapter V: Surfaces

**Source span.** Printed pages 356-423; PDF pages 371-438. The source is used only for structure, terminology, theorem orientation, and page spans. All prose, code, examples, and generated artifacts are original.

## Chapter Goal

This notebook studies nonsingular projective surfaces as objects controlled by intersection forms, birational transformations, and classification invariants. Curves had a single divisor degree; surfaces replace that with a bilinear intersection pairing. That one change is responsible for most of the chapter's geometry: Riemann-Roch on a surface, Hodge index, Nakai-Moishezon positivity, ruled surfaces, blowups, exceptional curves, cubic surfaces, and minimal models.

## Translation Guide

A divisor class on a surface is represented as a vector in a lattice, and the intersection product is represented as a symmetric matrix. A ruled surface is represented by two generators, a section and a fiber, whose intersection matrix controls positivity. A monoidal transformation is represented by adding an exceptional class with self-intersection `-1`, while strict transforms subtract multiplicity squared from self-intersection. A cubic surface is represented through the blowup of six points in the plane; the famous 27 lines become a finite incidence graph. Classification is represented as a geography chart organized by Kodaira dimension and birational invariants.

## Source Coverage Ledger

The source span was re-read before this revision. The notebook covers these section anchors: Geometry on a Surface; Ruled Surfaces; Monoidal Transformations; The Cubic Surface in P^3; Birational Transformations; Classification of Surfaces. The goal is not to reproduce the source, but to make the chapter's information inspectable as diagrams, matrices, exact checks, and small computational experiments.

## Library Routing

NumPy handles intersection matrices and grid tests. Matplotlib draws stable lattice, ruled-surface, blowup, and classification visuals. NetworkX is used for the cubic surface incidence graph because incidence is the central object, not a decorative network. Plotly provides an interactive classification lab where the learner can compare families and invariants. The generated files live under `artifacts/chapter-05/`.

## Visual Storyboard

1. Intersection-form dashboard for `P^2` blown up at two points.
2. Ruled-surface cone plot showing section/fiber coordinates and ampleness tests.
3. Monoidal transformation panel showing strict-transform self-intersection updates.
4. Cubic surface 27-line incidence graph from the six-point blowup model.
5. Birational classification geography lab organized by Kodaira dimension.


In [ ]:
from pathlib import Path
import sys, json, math
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import plotly.graph_objects as go

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "utils" / "artifacts.py").exists():
        BOOK_ROOT = candidate
        break
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, display_artifact, image_stats, save_json, save_matplotlib, save_plotly_html, save_table

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-05"
for child in ["figures", "html", "checks", "tables"]:
    (ARTIFACT_ROOT / child).mkdir(parents=True, exist_ok=True)

def relpath(path):
    return Path(path).resolve().relative_to(BOOK_ROOT).as_posix()

def intersection(d1, d2):
    a1, *b1 = d1
    a2, *b2 = d2
    return a1 * a2 - sum(x*y for x, y in zip(b1, b2))

source_sections = [
    {"number": "1", "title": "Geometry on a Surface", "printed_start": 357, "pdf_start": 372},
    {"number": "2", "title": "Ruled Surfaces", "printed_start": 369, "pdf_start": 384},
    {"number": "3", "title": "Monoidal Transformations", "printed_start": 386, "pdf_start": 401},
    {"number": "4", "title": "The Cubic Surface in P^3", "printed_start": 395, "pdf_start": 410},
    {"number": "5", "title": "Birational Transformations", "printed_start": 409, "pdf_start": 424},
    {"number": "6", "title": "Classification of Surfaces", "printed_start": 421, "pdf_start": 436},
]
visual_storyboard = [
    {"concept": "surface intersection form", "artifact": "figures/blowup-intersection-form-dashboard.png", "invariant": "matrix is symmetric with signature (1,2) in the sample basis"},
    {"concept": "ruled surface positivity", "artifact": "figures/ruled-surface-section-fiber-positivity.png", "invariant": "a>0 and b>a e in the displayed normalized ruled surface model"},
    {"concept": "monoidal transformation", "artifact": "figures/monoidal-transform-self-intersection-update.png", "invariant": "strict transform self-intersection equals C^2-m^2"},
    {"concept": "cubic surface lines", "artifact": "figures/cubic-surface-27-line-incidence-graph.png", "invariant": "27 vertices and degree 10 at every line in the Schlaefli model"},
    {"concept": "surface classification", "artifact": "html/surface-classification-geography-lab.html", "invariant": "families are organized by Kodaira dimension and birational markers"},
]
source_coverage_path = save_json({
    "chapter": "Chapter V: Surfaces",
    "printed_span": "356-423",
    "pdf_span": "371-438",
    "sections": source_sections,
    "copyright_note": "Original notebook prose and generated artifacts; source used only for orientation.",
}, ARTIFACT_ROOT, "checks", "source-coverage.json")
storyboard_path = save_json({"visual_sequence": visual_storyboard}, ARTIFACT_ROOT, "checks", "visual-storyboard.json")
generated_artifacts = [source_coverage_path, storyboard_path]
ARTIFACT_ROOT


In [ ]:
basis = ["H", "E1", "E2"]
M = np.diag([1, -1, -1])
classes = {
    "H": np.array([1, 0, 0]),
    "H-E1": np.array([1, 1, 0]),
    "H-E1-E2": np.array([1, 1, 1]),
    "2H-E1-E2": np.array([2, 1, 1]),
    "E1": np.array([0, -1, 0]),
    "E2": np.array([0, 0, -1]),
}
values = {name: int(v @ M @ v) for name, v in classes.items()}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.7))
im = axes[0].imshow(M, cmap="coolwarm", vmin=-1, vmax=1)
axes[0].set_xticks(range(3), basis); axes[0].set_yticks(range(3), basis)
for i in range(3):
    for j in range(3):
        axes[0].text(j, i, str(int(M[i,j])), ha="center", va="center", color="white" if abs(M[i,j]) > 0.5 else "black")
axes[0].set_title("Intersection matrix on Bl_{P1,P2} P2")
axes[1].barh(list(values), list(values.values()), color=["#1971c2" if v > 0 else "#d9480f" for v in values.values()])
axes[1].axvline(0, color="#333", lw=0.8)
axes[1].set_title("Sample self-intersections")
axes[1].set_xlabel("D^2")
fig.suptitle("Surface geometry begins with an intersection form", y=1.02)
form_path = save_matplotlib(fig, ARTIFACT_ROOT, "figures", "blowup-intersection-form-dashboard.png")
plt.close(fig)
generated_artifacts.append(form_path)
display_artifact(form_path, width=780)


The first surface visual replaces the curve degree with a lattice. In the blowup of the plane at two points, `H` has self-intersection `1`, while exceptional classes have self-intersection `-1`. The matrix is symmetric and indefinite, which is the numerical setting for the Hodge index theorem. The right panel gives concrete divisor classes so the learner can see how subtracting exceptional conditions changes self-intersection. This is the arithmetic behind strict transforms, exceptional curves, and positivity criteria.


In [ ]:
e = 2
avalues = np.arange(0, 7)
bvalues = np.arange(0, 15)
A, B = np.meshgrid(avalues, bvalues)
D2 = 2*A*B - e*A*A
ample = (A > 0) & (B > e*A) & (D2 > 0)
fig, axes = plt.subplots(1, 2, figsize=(11.3, 4.9))
for a in [1, 2, 3, 4, 5]:
    xs = np.linspace(0, 1, 20)
    axes[0].plot(xs + a, 0.25*np.sin(2*np.pi*xs) + 0.8*a, color="#1971c2", alpha=0.7)
axes[0].plot(np.linspace(0.5, 6.5, 100), 0.8*np.linspace(0.5, 6.5, 100)+0.8, color="#d9480f", lw=2.5, label="section C0")
axes[0].text(3.5, 0.35, "fibers f", ha="center")
axes[0].set_title("Ruled surface as section plus fibers")
axes[0].axis("off"); axes[0].legend(loc="upper left")
axes[1].imshow(ample.astype(int), origin="lower", aspect="auto", cmap="YlGn", extent=[avalues.min()-.5, avalues.max()+.5, bvalues.min()-.5, bvalues.max()+.5])
axes[1].plot(avalues, e*avalues, color="#9d0208", lw=2, label="b = e a")
axes[1].set_title("Ample region for D = aC0 + bf, e=2")
axes[1].set_xlabel("a"); axes[1].set_ylabel("b")
axes[1].legend(loc="upper left")
ruled_path = save_matplotlib(fig, ARTIFACT_ROOT, "figures", "ruled-surface-section-fiber-positivity.png")
plt.close(fig)
generated_artifacts.append(ruled_path)
display_artifact(ruled_path, width=780)


A ruled surface is a surface organized by a map to a curve whose fibers are projective lines. The notebook uses the standard section/fiber basis: `f^2=0`, `C0.f=1`, and `C0^2=-e`. The green grid marks divisor classes passing the displayed positivity test in a normalized model. This is a computational shadow of the chapter's detailed ruled-surface analysis: the geometry of curves on the surface is encoded by how they intersect the section and the fibers.


In [ ]:
initial_self = np.array([4, 3, 2, 1, 0, -1])
multiplicities = np.array([0, 1, 2, 1, 3, 2])
after = initial_self - multiplicities**2
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
idx = np.arange(len(initial_self))
axes[0].bar(idx - .18, initial_self, width=.36, label="before", color="#1971c2")
axes[0].bar(idx + .18, after, width=.36, label="strict transform", color="#f08c00")
axes[0].axhline(-1, color="#9d0208", ls="--", lw=1.4, label="exceptional threshold")
axes[0].set_xticks(idx, [f"m={m}" for m in multiplicities])
axes[0].set_ylabel("self-intersection")
axes[0].set_title("Blowup update C'² = C² - m²")
axes[0].legend()
G = nx.DiGraph()
G.add_edges_from([("surface X", "blow up point P"), ("blow up point P", "exceptional curve E"), ("exceptional curve E", "contract if E²=-1"), ("contract if E²=-1", "minimal model search")])
pos = {"surface X": (0,0), "blow up point P": (1.7,0.7), "exceptional curve E": (3.4,0), "contract if E²=-1": (5.1,0.7), "minimal model search": (6.8,0)}
nx.draw_networkx_edges(G, pos, ax=axes[1], arrows=True, arrowstyle="-|>", arrowsize=14, edge_color="#495057")
nx.draw_networkx_nodes(G, pos, ax=axes[1], node_color="#6741d9", node_size=1800)
nx.draw_networkx_labels(G, pos, ax=axes[1], font_color="white", font_size=8)
axes[1].set_title("Birational move: create or contract a -1 curve")
axes[1].axis("off")
blowup_path = save_matplotlib(fig, ARTIFACT_ROOT, "figures", "monoidal-transform-self-intersection-update.png")
plt.close(fig)
generated_artifacts.append(blowup_path)
display_artifact(blowup_path, width=780)


A monoidal transformation is the surface version of blowing up a point. The strict transform formula `C'^2 = C^2 - m^2` is the chapter's numerical heartbeat: passing through the center with multiplicity `m` lowers the self-intersection by `m^2`, and the exceptional curve itself has self-intersection `-1`. Castelnuovo's criterion then runs this arithmetic in the opposite direction: a rational curve with self-intersection `-1` can be contracted. The dependency graph shows why this becomes a factorization language for birational maps of surfaces.


In [ ]:
def E(i):
    b = [0]*6; b[i] = -1
    return tuple([0] + b)
def L(i, j):
    b = [0]*6; b[i] = 1; b[j] = 1
    return tuple([1] + b)
def Q(i):
    b = [1]*6; b[i] = 0
    return tuple([2] + b)
line_classes = {}
for i in range(6):
    line_classes[f"E{i+1}"] = E(i)
for i in range(6):
    for j in range(i+1, 6):
        line_classes[f"L{i+1}{j+1}"] = L(i, j)
for i in range(6):
    line_classes[f"Q{i+1}"] = Q(i)
G = nx.Graph()
G.add_nodes_from(line_classes)
for a, ca in line_classes.items():
    for b, cb in line_classes.items():
        if a < b and intersection(ca, cb) == 1:
            G.add_edge(a, b)
degrees = dict(G.degree())
fig, ax = plt.subplots(figsize=(9.2, 9.2))
pos = nx.circular_layout(G)
node_colors = ["#1971c2" if n.startswith("E") else "#f08c00" if n.startswith("L") else "#6741d9" for n in G.nodes]
nx.draw_networkx_edges(G, pos, ax=ax, width=0.7, alpha=0.35, edge_color="#495057")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=360)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=7, font_color="white")
ax.set_title("Cubic surface: 27 lines as an incidence graph")
ax.axis("off")
cubic_path = save_matplotlib(fig, ARTIFACT_ROOT, "figures", "cubic-surface-27-line-incidence-graph.png")
plt.close(fig)
generated_artifacts.append(cubic_path)
display_artifact(cubic_path, width=760)


The cubic surface visual is a finite model of a classical theorem. Blowing up six points of the plane gives 27 divisor classes: six exceptional curves, fifteen transforms of lines through pairs of points, and six transforms of conics through five of the points. Two lines meet when their intersection product is one. The graph check below verifies that every vertex has degree ten, which is the expected incidence count for the 27-line configuration. This is a better notebook representation than a copied source figure because the learner can inspect and recompute the incidence relation.


In [ ]:
classification = [
    {"family": "rational surfaces", "kodaira": -1, "p_g": 0, "K2": 9, "marker": "P2 and ruled models"},
    {"family": "ruled over genus g", "kodaira": -1, "p_g": 0, "K2": 8, "marker": "minimal except rational exceptions"},
    {"family": "K3 surface", "kodaira": 0, "p_g": 1, "K2": 0, "marker": "trivial canonical class"},
    {"family": "abelian surface", "kodaira": 0, "p_g": 1, "K2": 0, "marker": "group variety"},
    {"family": "elliptic surface", "kodaira": 1, "p_g": 2, "K2": 0, "marker": "fibration controls pluricanonical image"},
    {"family": "general type", "kodaira": 2, "p_g": 4, "K2": 6, "marker": "pluricanonical map birational"},
]
fig_html = go.Figure()
fig_html.add_trace(go.Scatter(
    x=[item["kodaira"] for item in classification],
    y=[item["K2"] for item in classification],
    mode="markers+text",
    text=[item["family"] for item in classification],
    textposition="top center",
    marker={"size": [14 + 5*item["p_g"] for item in classification], "color": [item["p_g"] for item in classification], "colorscale": "Plasma", "showscale": True, "colorbar": {"title": "p_g"}},
    customdata=[item["marker"] for item in classification],
    hovertemplate="%{text}<br>Kodaira dimension %{x}<br>K^2 %{y}<br>%{customdata}<extra></extra>",
))
fig_html.update_layout(title="Surface classification geography lab", xaxis_title="Kodaira dimension", yaxis_title="sample K^2", template="plotly_white", height=540)
classification_path = save_plotly_html(fig_html, ARTIFACT_ROOT, "html", "surface-classification-geography-lab.html")
generated_artifacts.append(classification_path)
display_artifact(classification_path, width="100%", height=420)


The classification lab is intentionally a geography chart rather than a complete classification theorem. The chapter explains why surfaces differ from curves: a birational class can have many nonsingular models, so one first removes contractible `-1` curves and works with relatively minimal models. Kodaira dimension then organizes the remaining terrain. Rational and ruled surfaces sit at `-1`, special zero-dimensional canonical behavior sits at `0`, elliptic fibrations occupy `1`, and general type surfaces occupy `2`. The chart makes those categories visible without pretending the numerical invariants listed here are a complete moduli theory.


In [ ]:
intersection_symmetric = bool(np.allclose(M, M.T))
intersection_eigenvalues = np.linalg.eigvalsh(M).round(8).tolist()
ruled_ample_samples = [
    {"a": int(a), "b": int(b), "D2": int(2*a*b - e*a*a), "ample_model": bool((a > 0) and (b > e*a) and (2*a*b - e*a*a > 0))}
    for a, b in [(1, 3), (2, 5), (3, 5), (0, 4)]
]
blowup_rows = [{"C2_before": int(c), "multiplicity": int(m), "C2_after": int(a)} for c, m, a in zip(initial_self, multiplicities, after)]
cubic_degree_values = sorted(set(degrees.values()))
source_table_path = save_table(source_sections, ARTIFACT_ROOT, "tables", "chapter-05-source-coverage.csv")
checks_path = save_json({
    "intersection_matrix_symmetric": intersection_symmetric,
    "intersection_eigenvalues": intersection_eigenvalues,
    "ruled_ample_samples": ruled_ample_samples,
    "blowup_self_intersection_updates": blowup_rows,
    "cubic_line_vertex_count": G.number_of_nodes(),
    "cubic_line_edge_count": G.number_of_edges(),
    "cubic_line_degrees": cubic_degree_values,
    "classification_families": classification,
}, ARTIFACT_ROOT, "checks", "surface-invariant-checks.json")
generated_artifacts.extend([source_table_path, checks_path])
assert intersection_symmetric
assert intersection_eigenvalues == [-1.0, -1.0, 1.0]
assert all(row["C2_after"] == row["C2_before"] - row["multiplicity"]**2 for row in blowup_rows)
assert G.number_of_nodes() == 27
assert cubic_degree_values == [10]
assert_artifacts(generated_artifacts)
records = []
for artifact in generated_artifacts:
    record = {"path": relpath(artifact), "exists": artifact.exists(), "bytes": artifact.stat().st_size}
    if artifact.suffix.lower() == ".png":
        record.update(image_stats(artifact))
    records.append(record)
final_sanity = {
    "chapter": "Chapter V: Surfaces",
    "source_span": {"printed": "356-423", "pdf": "371-438"},
    "artifacts": records,
    "checks": {
        "intersection_form_signature_sample": intersection_eigenvalues,
        "ruled_surface_ample_grid_computed": True,
        "monoidal_self_intersection_formula_checked": True,
        "cubic_surface_27_lines_degree_10": True,
        "classification_lab_records_kodaira_dimension": True,
    },
    "standalone_contract": "original prose, generated visuals, and local checks; no copied source text or figures",
}
final_path = save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")
final_sanity["artifacts"].append({"path": relpath(final_path), "exists": final_path.exists(), "bytes": final_path.stat().st_size})
save_json(final_sanity, ARTIFACT_ROOT, "checks", "final-sanity.json")
final_sanity


## Standalone Study Notes

The surface chapter should be read as the moment when geometry becomes two-dimensional enough that a single degree no longer controls the story. Intersection numbers replace degree, and the intersection form keeps track of how divisor classes meet. Riemann-Roch for surfaces uses that form together with the canonical class and arithmetic genus. Ruled surfaces make the lattice concrete because every divisor can be compared with a section and a fiber. Monoidal transformations show how the lattice changes under a blowup: the exceptional curve introduces a new negative direction, and strict transforms lose multiplicity squared. Cubic surfaces then become a beautiful finite laboratory where the same intersection arithmetic produces 27 lines and their incidence. Birational transformations use blowups and contractions to factor maps, and classification asks what remains after contractible curves are removed.

The notebook's checks are deliberately numerical. The matrix signature is the local language of Hodge index. The ruled-surface grid turns ampleness into inequalities. The blowup panel tests the strict-transform formula. The cubic graph verifies the 27-line incidence count. The classification lab keeps the minimal-model viewpoint visible without pretending to replace the deep classification theory.


## Takeaways

Surface theory is governed by the tension between local modifications and global numerical invariants. Intersection pairings turn divisor classes into a lattice. Ruled surfaces show how a fibration over a curve makes this lattice concrete. Blowups add exceptional curves and change self-intersections in a controlled way; contractions reverse that move when Castelnuovo's criterion applies. Cubic surfaces compress this arithmetic into a finite incidence configuration of 27 lines. Classification then asks which birational models remain after all contractible curves have been removed, and how canonical divisors organize the result. The notebook's checks keep the visuals honest: matrices are symmetric, blowup formulas balance, the cubic graph has the expected degree, and every artifact is local to the course.
